<a href="https://colab.research.google.com/github/nir-nutman/RapidFire-AI-LLM-Experimentation/blob/v2-corrected-metrics/notebooks/UCSB_catalog_rag_notebook_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UCSB Course Catalog RAG System — v2 (corrected metrics)
## RapidFire AI Winter Competition - RAG Track

**This is NOT the competition submission.** The original submitted notebook is unchanged at
`notebooks/UCSB_catalog_rag_notebook.ipynb` (frozen at git tag `pre-correction`). This v2
notebook is a separate, post-competition correction pass that fixes bugs found in the metrics
code during a later audit. It has not been run yet — cells below are ready to execute in Colab,
but no results exist until you run them.

**What changed from the original notebook, and why — see `README.md`'s "Post-competition
corrections" section on this branch for full detail. Summary:**
1. `compute_ndcg_at_k` used an ideal relevance score of 3 while actual hits were scored 1,
   capping the maximum achievable NDCG@5 at 1/3 regardless of retrieval quality. Fixed to 1.
2. `sample_compute_metrics_fn` converted `retrieved_documents` to a Python `set` before
   computing NDCG@5 and MRR, destroying the retrieval rank order those metrics need. Fixed to
   de-duplicate while preserving order (`list(dict.fromkeys(pred))`).
3. Precision/Recall/F1 are now explicitly documented as being computed over **unique source
   sections (`corpus_id`) after de-duplication** — because several retrieved chunks can share
   one `corpus_id` (see README), not over the raw per-chunk retrieval list.
4. Per-query ordered retrieved-ID lists are now saved to `results/retrieved_ids_v2.json` so the
   numbers can be audited later without re-running the GPU job.

**Dataset:** UCSB 2009-2010 Course Catalog
**Research Question:** Which chunking and reranking strategies optimize retrieval for course catalog queries?
**Configurations Tested:** 4 (chunk_size: [128, 256] × reranking top_n: [2, 5])

## Experiment Hypothesis

Larger chunks (256 tokens) will better preserve course description context compared to smaller chunks (128 tokens), leading to improved Precision and NDCG@5 scores.
Keeping more reranked documents (top_n=5) will improve Recall at the cost of some Precision.

**Note on top_n:** an audit of the installed `rapidfireai` source (see README) confirmed
`rag.get_context` does apply the cross-encoder reranker and the configured `top_n` cutoff —
top_n is not silently ignored. The original run's byte-identical metrics across top_n=2 vs
top_n=5 are most plausibly explained by bug #2 above (the same source-document chunks surviving
either cutoff, then collapsing to the same unique `corpus_id` set) — not by top_n being unused.
This v2 run will show whether that explanation holds once de-duplication is made explicit rather
than accidental.

In [ ]:
try:
    import rapidfireai
    installed_version = getattr(rapidfireai, "__version__", "unknown")
    assert installed_version == "0.12.9", (
        f"Expected rapidfireai==0.12.9 (the version live on PyPI during the original "
        f"2026-01-21 run), found {installed_version}. The RFLangChainRagSpec constructor's "
        f"parameter names changed in later releases (e.g. embedding_cls/vector_store/"
        f"search_type/search_kwargs/reranker_cls/reranker_kwargs here vs embedding_cfg/"
        f"vector_store_cfg/search_cfg/reranker_cfg on current main) and this notebook will "
        f"fail with TypeErrors on an unpinned/newer install. Uninstall and reinstall 0.12.9."
    )
    print("✅ rapidfireai 0.12.9 already installed")
except ImportError:
    %pip install rapidfireai==0.12.9
    !rapidfireai init --evals

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ['PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION'] = 'python'

from rapidfireai import Experiment
from rapidfireai.automl import List, RFLangChainRagSpec, RFvLLMModelConfig, RFPromptManager, RFGridSearch
import re, json
from typing import List as listtype, Dict, Any

## 📊 Dataset Preparation

The UCSB 2009-2010 catalog corpus, queries, and relevance judgments ship in this repository's
`data/` folder — **not** under a separate tutorial_notebooks path. (The original notebook
referenced `/content/tutorial_notebooks/rag-contexteng/datasets/ucsb_catalog/...`, a path that
only exists if RapidFire AI's own tutorial assets happen to already be present in the Colab
runtime; it was never reproducible from a clean Colab session or from this repo alone. v2 clones
this repo directly instead, so the notebook is self-contained.)

- **corpus.jsonl**: 1,304 source-section records extracted from the 490-page PDF. **Each record is
  a pre-chunking document** (mean length ~3,499 characters) — the pipeline's own text_splitter
  (chunk_size 128/256 tokens) subdivides most of these further at index time. "1,304" is the count
  of source sections, not the count of retrieval-time chunks.
- **queries.jsonl**: 30 typical student questions about courses and requirements
- **qrels.tsv**: 150 relevance judgments (5 per query). Per the original README/notebook, these
  were "generated via embedding similarity" — no data-preparation script for this exists anywhere
  in this repository's git history, so the exact method, embedding model, and whether any human
  review occurred could not be verified by audit. Treat the ground truth itself as unverified.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nir-nutman/RapidFire-AI-LLM-Experimentation.git"
REPO_DIR = Path("/content/RapidFire-AI-LLM-Experimentation")

if not REPO_DIR.exists():
    # Clone the v2-corrected-metrics branch specifically, so the cloned data/ matches this notebook.
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "v2-corrected-metrics", REPO_URL, str(REPO_DIR)],
        check=True,
    )

dataset_dir = REPO_DIR / "data"
for fname in ("corpus.jsonl", "queries.jsonl", "qrels.tsv"):
    assert (dataset_dir / fname).exists(), f"{fname} not found under {dataset_dir} — clone may have failed."
print(f"✅ Repo data staged at {dataset_dir}")

# Where v2 outputs get written back into the repo checkout, so they can be copied/committed afterward.
results_dir = REPO_DIR / "results"
results_dir.mkdir(exist_ok=True)

In [ ]:
from datasets import load_dataset
import pandas as pd
import random

# Load UCSB catalog queries
ucsb_dataset = load_dataset(
    "json",
    data_files=str(dataset_dir / "queries.jsonl"),
    split="train"
)

# Load relevance labels
qrels = pd.read_csv(str(dataset_dir / "qrels.tsv"), sep="\t")

# Sample 50% of queries for this experiment (unchanged from original, for comparability)
sample_fraction = 0.5
rseed = 1
random.seed(rseed)

sample_size = int(len(ucsb_dataset) * sample_fraction)
ucsb_dataset = ucsb_dataset.shuffle(seed=rseed).select(range(sample_size))

query_ids = set([int(qid) for qid in ucsb_dataset["query_id"]])
qrels = qrels[qrels["query_id"].isin(query_ids)]

print(f"Using {len(ucsb_dataset)} queries ({sample_fraction*100}% of dataset)")
print(f"Filtered qrels to {len(qrels)} relevance judgments")

## 🔧 RAG Pipeline Configuration

Testing 4 configurations (unchanged from the original — only the metrics code below is being
corrected, not the retrieval/reranking configuration itself):

| Config | Chunk Size | Rerank Top-N | Hypothesis |
|--------|-----------|--------------|------------|
| 1 | 256 | 2 | High precision, focused context |
| 2 | 256 | 5 | Balanced precision & recall |
| 3 | 128 | 2 | Granular chunks, focused results |
| 4 | 128 | 5 | Granular chunks, broad coverage |

**Fixed Parameters:**
- Embedding: sentence-transformers/all-MiniLM-L6-v2
- Initial retrieval: k=8 (similarity search)
- Reranker: cross-encoder/ms-marco-MiniLM-L6-v2
- Generator: Qwen/Qwen2.5-0.5B-Instruct

Confirmed by reading the installed `rapidfireai==0.12.9` source
(`rapidfireai/evals/rag/rag_pipeline.py`): with `vector_store=None` and
`enable_gpu_search=True`, `LangChainRagSpec.build_pipeline()` builds a **FAISS** index using
`faiss.IndexFlatL2` moved to GPU (exact L2 distance search, not approximate). With
`enable_gpu_search=False` it would instead build `faiss.IndexHNSWFlat` (approximate, CPU).

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

batch_size = 50

rag_gpu = RFLangChainRagSpec(
    document_loader=DirectoryLoader(
        path=str(dataset_dir),
        glob="corpus.jsonl",
        loader_cls=JSONLoader,
        loader_kwargs={
            "jq_schema": ".",
            "content_key": "text",
            "metadata_func": lambda record, metadata: {
                "corpus_id": int(record.get("_id"))
            },
            "json_lines": True,
            "text_content": False,
        },
        sample_seed=42,
    ),
    # EXPERIMENT VARIABLE 1: Chunk size
    text_splitter=List([
        RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2", chunk_size=256, chunk_overlap=32
        ),
        RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            encoding_name="gpt2", chunk_size=128, chunk_overlap=32
        ),
    ]),
    embedding_cls=HuggingFaceEmbeddings,
    embedding_kwargs={
        "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        "model_kwargs": {"device": "cuda:0"},
        "encode_kwargs": {"normalize_embeddings": True, "batch_size": batch_size},
    },
    vector_store=None,
    search_type="similarity",
    search_kwargs={"k": 8},
    # EXPERIMENT VARIABLE 2: Reranking strategy
    reranker_cls=CrossEncoderReranker,
    reranker_kwargs={
        "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
        "model_kwargs": {"device": "cpu"},
        "top_n": List([2, 5]),
    },
    enable_gpu_search=True,
)

In [ ]:
def sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Retrieve context and format prompts for LLM.

    Unchanged from the original notebook: retrieved_documents here is already in true
    retrieval-rank order (it's built directly from rag.get_context's returned Document list,
    before any set conversion). The ordering bug this v2 fixes was downstream, in
    sample_compute_metrics_fn — see that cell.
    """

    INSTRUCTIONS = """You are a helpful academic advisor for UCSB students.
Use the provided course catalog information to answer questions about courses,
requirements, policies, and academic programs. Be specific and cite relevant
catalog sections when possible."""

    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)

    retrieved_documents = [
        [doc.metadata["corpus_id"] for doc in docs] for docs in all_context
    ]

    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": f"""Here is relevant information from the UCSB course catalog:

{context}

Based on this catalog information, please answer the following question:
{question}""",
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "retrieved_documents": retrieved_documents,
        **batch,
    }

In [ ]:
def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Attach ground truth documents for evaluation"""
    batch["ground_truth_documents"] = [
        qrels[qrels["query_id"] == query_id]["corpus_id"].tolist()
        for query_id in batch["query_id"]
    ]
    return batch

## 🔧 Metrics fixes (v2)

**What Precision, Recall, and F1 are computed over:** unique source sections (`corpus_id`),
after de-duplication of the retrieved list — not the raw per-chunk retrieval list. This project's
RAG config does not expose a finer chunk-level ID: `metadata_func` (in the RAG pipeline config
cell above) only attaches `corpus_id` from the parent source record, and the text splitter isn't
configured with `add_start_index`, so no per-sub-chunk identifier survives into the Document
metadata LangChain's `get_context` returns. **Chunk-level Precision is not reported here because
the framework/configuration as currently set up doesn't expose chunk-level IDs** — adding one
would mean changing the RAG pipeline configuration itself, which is out of scope for this
metrics-only correction pass.

**What changed below, and why** (see README's "Post-competition corrections" section for the
full audit writeup):
- `compute_ndcg_at_k`: ideal relevance changed from `3` to `1`, matching the `1` used for actual
  hits. The original mismatch capped the maximum achievable NDCG@5 at 1/3 regardless of how good
  retrieval was.
- `sample_compute_metrics_fn`: `retrieved_documents` is de-duplicated with
  `list(dict.fromkeys(pred))` (preserves first-seen order) instead of `set(pred)` (which
  discards order entirely). The de-duplicated **ordered** list is passed to `compute_ndcg_at_k`
  and `compute_rr`, so rank position is preserved for the metrics that depend on it. Precision/
  Recall still use the de-duplicated set (as before), now with that explicitly documented rather
  than incidental.

In [ ]:
import math

def compute_ndcg_at_k(retrieved_docs: list, expected_docs: set, k=5):
    """Compute NDCG@k metric.

    v2 fix: ideal relevance is now 1 (matching actual-hit relevance of 1), not 3. The original
    3-vs-1 mismatch meant IDCG = 3x the best possible DCG, capping max achievable NDCG@5 at 1/3.

    v2 fix: retrieved_docs is now an ORDERED list (caller passes a de-duplicated, rank-preserving
    list, not a set) — this function was always rank-sensitive (it slices [:k] and weights by
    position), it just wasn't being given real order before.
    """
    relevance = [1 if doc in expected_docs else 0 for doc in list(retrieved_docs)[:k]]
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))

    ideal_length = min(k, len(expected_docs))
    ideal_relevance = [1] * ideal_length + [0] * (k - ideal_length)  # v2: was [3] * ideal_length
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0.0

def compute_rr(retrieved_docs: list, expected_docs: set):
    """Compute Reciprocal Rank for a single query.

    v2: retrieved_docs is now an ordered list (see sample_compute_metrics_fn) instead of a set,
    so "first relevant doc" is actually the first one retrieved, not an arbitrary set-iteration
    order.
    """
    rr = 0
    for i, retrieved_doc in enumerate(retrieved_docs):
        if retrieved_doc in expected_docs:
            rr = 1 / (i + 1)
            break
    return rr

def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Compute retrieval metrics per batch"""
    precisions, recalls, f1_scores, ndcgs, rrs = [], [], [], [], []
    total_queries = len(batch["query"])

    for pred, gt in zip(batch["retrieved_documents"], batch["ground_truth_documents"]):
        expected_set = set(gt)
        # v2 fix: de-duplicate corpus_ids while preserving retrieval rank order, instead of
        # set(pred) (which discards order entirely — NDCG/MRR need order, and several retrieved
        # chunks can legitimately share one corpus_id, per the dataset audit in the README).
        retrieved_ordered = list(dict.fromkeys(pred))
        retrieved_set = set(retrieved_ordered)

        true_positives = len(expected_set.intersection(retrieved_set))
        precision = true_positives / len(retrieved_set) if len(retrieved_set) > 0 else 0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        # v2 fix: pass the ORDERED de-duplicated list, not the unordered set.
        ndcgs.append(compute_ndcg_at_k(retrieved_ordered, expected_set, k=5))
        rrs.append(compute_rr(retrieved_ordered, expected_set))

    return {
        "Total": {"value": total_queries},
        "Precision": {"value": sum(precisions) / total_queries},
        "Recall": {"value": sum(recalls) / total_queries},
        "F1 Score": {"value": sum(f1_scores) / total_queries},
        "NDCG@5": {"value": sum(ndcgs) / total_queries},
        "MRR": {"value": sum(rrs) / total_queries},
    }

def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Accumulate metrics across all batches (unchanged from original)"""
    num_queries_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total_queries = sum(num_queries_per_batch)
    algebraic_metrics = ["Precision", "Recall", "F1 Score", "NDCG@5", "MRR"]

    return {
        "Total": {"value": total_queries},
        **{
            metric: {
                "value": sum(
                    m["value"] * queries
                    for m, queries in zip(
                        aggregated_metrics[metric], num_queries_per_batch
                    )
                ) / total_queries,
                "is_algebraic": True,
                "value_range": (0, 1),
            }
            for metric in algebraic_metrics
        },
    }

In [ ]:
vllm_config1 = RFvLLMModelConfig(
    model_config={
        "model": "Qwen/Qwen2.5-0.5B-Instruct",
        "dtype": "half",
        "gpu_memory_utilization": 0.25,
        "tensor_parallel_size": 1,
        "distributed_executor_backend": "mp",
        "enable_chunked_prefill": False,
        "enable_prefix_caching": False,
        "max_model_len": 3000,
        "disable_log_stats": True,
        "enforce_eager": True,
        "disable_custom_all_reduce": True,
    },
    sampling_params={
        "temperature": 0.8,
        "top_p": 0.95,
        "max_tokens": 128,
    },
    rag=rag_gpu,
    prompt_manager=None,
)

batch_size = 3

config_set = {
    "vllm_config": vllm_config1,
    "batch_size": batch_size,
    "preprocess_fn": sample_preprocess_fn,
    "postprocess_fn": sample_postprocess_fn,
    "compute_metrics_fn": sample_compute_metrics_fn,
    "accumulate_metrics_fn": sample_accumulate_metrics_fn,
    "online_strategy_kwargs": {
        "strategy_name": "normal",
        "confidence_level": 0.95,
        "use_fpc": True,
    },
}

In [ ]:
config_group = RFGridSearch(config_set)

## 🚀 Running Experiments (v2)

This cell executes all 4 configurations in parallel using RapidFire AI's experiment orchestration
system, using the corrected metrics code above. Uses a separate `experiment_name` from the
original run so it doesn't collide with any existing RapidFire experiment state for
`ucsb-catalog-rag-exp1`.

**Expected Runtime:** ~20-30 minutes on Colab T4 GPU (unchanged — only metrics scoring changed,
not retrieval/generation cost).

In [ ]:
experiment = Experiment(experiment_name="ucsb-catalog-rag-exp1-v2", mode="evals")

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8855)

In [ ]:
results = experiment.run_evals(
    config_group=config_group,
    dataset=ucsb_dataset,
    num_actors=1,
    num_shards=4,
    seed=42,
)

## 📊 Results Analysis (v2)

The table below shows the corrected final metrics for all 4 configurations.

**Key Metrics (same definitions as original, Precision/Recall/F1 now explicitly over
de-duplicated unique `corpus_id`, NDCG@5/MRR now rank-order-aware):**
- **Precision**: Of unique retrieved source sections, what % were relevant?
- **Recall**: Of all relevant source sections, what % were retrieved?
- **F1 Score**: Harmonic mean of Precision and Recall
- **NDCG@5**: How well were relevant documents ranked? (0-1 scale, 1 = perfect ranking — v2 fix
  removes the old 1/3 ceiling)
- **MRR**: How quickly was the first relevant document found, in true retrieval rank order (v2 fix)

In [ ]:
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v
     for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

# Display key columns only for clarity
display_cols = ['run_id', 'chunk_size', 'top_n', 'Precision', 'Recall', 'F1 Score', 'NDCG@5', 'MRR']
display(results_df[display_cols])

# --- v2 additions below: save corrected CSV into the repo checkout, and save per-query ---
# --- ordered retrieved-ID lists so the numbers can be audited later without a GPU re-run. ---

v2_csv_path = results_dir / "ucsb_catalog_results_v2.csv"
results_df.to_csv(v2_csv_path, index=False)
print(f"✅ Corrected results saved to {v2_csv_path}")

# aggregated_results (the first element of each results[run_id] tuple) is built by RapidFire's
# own batch aggregator from whatever sample_preprocess_fn/sample_postprocess_fn returned per
# batch — confirmed by reading rapidfireai==0.12.9's evals/metrics/aggregator.py and
# experiment.py (run_evals docstring: "Dict mapping run_id to (aggregated_results,
# cumulative_metrics) tuple"). It was never actually executed here, so this is verified from
# source, not from having run it — double-check the printed keys below before trusting the export.
retrieved_ids_v2 = {}
for run_id, (aggregated_results, metrics_dict) in results.items():
    print(f"run {run_id}: aggregated_results keys = {list(aggregated_results.keys())}")
    if "query_id" not in aggregated_results or "retrieved_documents" not in aggregated_results:
        print(f"⚠️  run {run_id}: expected keys not found — skipping retrieved_ids export for this run.")
        continue
    per_query = {}
    for qid, pred in zip(aggregated_results["query_id"], aggregated_results["retrieved_documents"]):
        # Same de-duplication as sample_compute_metrics_fn, kept consistent with what was scored.
        per_query[str(qid)] = list(dict.fromkeys(pred))
    retrieved_ids_v2[str(run_id)] = {
        "chunk_size": metrics_dict.get("chunk_size"),
        "top_n": metrics_dict.get("top_n"),
        "retrieved_ids_by_query": per_query,
    }

retrieved_ids_path = results_dir / "retrieved_ids_v2.json"
with open(retrieved_ids_path, "w") as f:
    json.dump(retrieved_ids_v2, f, indent=2)
print(f"✅ Per-query ordered retrieved IDs saved to {retrieved_ids_path}")

## 🔍 Key Findings (v2) — fill in after running the cells above

**This section is intentionally left as a template.** Nothing in this notebook has been executed
by Claude — I don't have a GPU runtime here, and I was explicitly told not to guess at results.
Run the cells above in Colab, then come back and replace the bracketed placeholders with what
`results_df` and `retrieved_ids_v2.json` actually show.

### Best Configuration: [fill in from results_df after running]

**Corrected metrics:**
- **Precision**: [value] (now over de-duplicated unique `corpus_id`)
- **Recall**: [value]
- **F1 Score**: [value]
- **NDCG@5**: [value] (0-1 scale now, not capped at 1/3)
- **MRR**: [value] (now computed on true rank order)

### Observations — fill in only what the numbers actually show

- Chunk size impact: [compare 256 vs 128 using the new numbers — do NOT reuse the old notebook's
  "16-21%" / "20% faster" / "9% more accurate" language unless the new numbers independently
  support an equivalent, correctly-labeled statement]
- Reranking top_n impact: [compare top_n=2 vs top_n=5 now that order and de-duplication are
  fixed — check whether they're still identical; if so, that's informative in itself, see README]
- Anything NDCG@5/MRR show now that they couldn't reliably show before

### What NOT to carry over from the original Key Findings section

The following claims from the original notebook were flagged in audit as unsupported by any code
in this repository, and should not be restated here even if the new numbers happen to look
similar: "students find information 20% faster," "answers are 9% more accurate," and "real-time
confidence intervals showed superiority after 50% of data was processed" (not substantiated by
`logs/rapidfire.log`). MRR and Precision are retrieval metrics, not measures of user-perceived
speed or generated-answer correctness — if you want to make a speed or accuracy claim, it needs
its own measurement, not a relabeling of these metrics.

In [ ]:
experiment.end()
print("v2 experiment completed")

In [ ]:
# Save to Google Drive for backup, same as original notebook
results_df.to_csv('/content/drive/MyDrive/ucsb_catalog_results_v2.csv', index=False)
print("✅ Results also saved to Google Drive")

print()
print("Files now inside the repo checkout at", REPO_DIR, ":")
print(" -", v2_csv_path)
print(" -", retrieved_ids_path)
print()
print("To get them into your actual local clone / the open PR, from a terminal:")
print(f"  cd {REPO_DIR} && git status")
print("  # then download/copy results/ucsb_catalog_results_v2.csv and")
print("  # results/retrieved_ids_v2.json from this Colab runtime to your machine,")
print("  # or git add/commit/push directly from this Colab cell if you've set up a token.")